# Stage 3a — Extracting diffs for the Levin commits

**Purpose.** Retrieve the source-code change for each of the labelled commits, so Stage 4 can train
CodeBERT on message **plus** diff.

**Why this exists.** Levin's second model half uses ChangeDistiller change types, which cannot be
recomputed on modern Java — the tool dates from 2006 and its parser predates lambdas, records and
sealed classes, so failures would be silent and uneven across projects. The diff is the raw material
that taxonomy was derived from, and a code language model can read it directly.

**Why the API rather than cloning.** Eleven repositories including intellij-community and
elasticsearch cost upwards of ten gigabytes and hours to clone, even with `--filter=blob:none`.
GitHub's commits endpoint returns the patch per file directly, and 1,149 commits sits well inside
one hour of authenticated quota.

**What will not be recoverable.** These are 2017-era hashes. Repositories move, histories get
rewritten, and force-pushes drop commits. Some fraction will 404, and that fraction is reported
rather than quietly dropped — it becomes a stated limitation on how much of the labelled set Stage 4
can use.

## 0. Setup

Run this cell first. It works in Google Colab, Jupyter and IDEs such as VS Code or PyCharm.

It finds this repository (on Colab, opened straight from GitHub, it clones it), then picks one
working folder, `WORK`, where this notebook reads its inputs and writes its outputs. Every
notebook in the repository uses the same `WORK`, so they hand files to each other through it.

- set the environment variable `CIC_WORK` to choose the folder yourself
- otherwise it is `<repository>/work`, or on Colab a folder on your Google Drive
  (set `CIC_DRIVE=0` to stay on Colab's local disk)

An input missing from `WORK` is copied from the repository or downloaded from this study's Zenodo
record or Hugging Face repository, so any notebook can be run without re-running the ones before
it. See `replication.py`.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO = "https://github.com/Binamra00/commit-intent-and-refactoring-levels"
_candidates = [Path(os.environ["CIC_ROOT"])] if os.environ.get("CIC_ROOT") else []
_candidates += [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in _candidates if (p / "replication.py").exists()), None)
if ROOT is None:                      # opened on its own, e.g. from GitHub in Colab
    ROOT = Path.cwd() / REPO.rsplit("/", 1)[-1]
    if not (ROOT / "replication.py").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO, str(ROOT)], check=True)
sys.path.insert(0, str(ROOT))

from replication import setup, require
WORK = setup()

## 1. Inputs and the project map

The `project` column does not name GitHub repositories directly — `ReactiveX-RxJava` is one string
for owner and repo, `hbase` omits the `apache` owner entirely. So the mapping is written out
explicitly rather than derived, and each entry is verified against the API before any commit is
fetched.

In [2]:
import json, os, time, getpass, collections, urllib.request, urllib.error
from pathlib import Path
import pandas as pd

need = ["levin_dataset.pkl", "levin_columns.json"]
missing = [f for f in need if not Path(f).exists()]
if missing:
    raise SystemExit(f"missing {missing} -- run Stage 1 first, in this directory")

D = pd.read_pickle("levin_dataset.pkl")
COLS = json.load(open("levin_columns.json"))
ID, PROJECT, MESSAGE, LABEL = COLS["hash"], COLS["project"], COLS["message"], COLS["label"]

print(f"{len(D):,} commits")
print(f"\nprojects as they appear in the dataset:\n")
print(D[PROJECT].value_counts().to_string())

1,151 commits

projects as they appear in the dataset:

project
drools                    114
hadoop                    112
hbase                     109
intellij-community        109
camel                     103
orientdb                  102
kotlin                    101
restlet-framework-java    101
ReactiveX-RxJava          100
elasticsearch             100
spring-framework          100


In [3]:
# project name in the dataset -> owner/repo on GitHub.
# Written out rather than derived: the column mixes forms, and several of these projects have
# moved owners since 2017. Each is verified in the next cell.
PROJECT_REPO = {
    "ReactiveX-RxJava":       "ReactiveX/RxJava",
    "RxJava":                 "ReactiveX/RxJava",
    "hbase":                  "apache/hbase",
    "elasticsearch":          "elastic/elasticsearch",
    "intellij-community":     "JetBrains/intellij-community",
    "hadoop":                 "apache/hadoop",
    "drools":                 "apache/incubator-kie-drools",
    "kotlin":                 "JetBrains/kotlin",
    "restlet-framework-java": "restlet/restlet-framework-java",
    "orientdb":               "orientechnologies/orientdb",
    "camel":                  "apache/camel",
    "spring-framework":       "spring-projects/spring-framework",
}

unmapped = sorted(set(D[PROJECT]) - set(PROJECT_REPO))
if unmapped:
    print(f"NOT MAPPED: {unmapped}")
    print("Add them above before continuing -- their commits cannot be fetched.")
else:
    print("every project in the dataset has a mapping")
    for p in sorted(D[PROJECT].unique()):
        print(f"   {p:<26}-> {PROJECT_REPO[p]}")

every project in the dataset has a mapping
   ReactiveX-RxJava          -> ReactiveX/RxJava
   camel                     -> apache/camel
   drools                    -> apache/incubator-kie-drools
   elasticsearch             -> elastic/elasticsearch
   hadoop                    -> apache/hadoop
   hbase                     -> apache/hbase
   intellij-community        -> JetBrains/intellij-community
   kotlin                    -> JetBrains/kotlin
   orientdb                  -> orientechnologies/orientdb
   restlet-framework-java    -> restlet/restlet-framework-java
   spring-framework          -> spring-projects/spring-framework


### Token

Unauthenticated requests are capped at 60 per hour, which is not enough for 1,149 commits.
A token raises that to 5,000. It is requested at runtime and never written to disk.

A fine-grained token with no scopes, or a classic token with `public_repo`, is sufficient — these
are all public repositories. Set the `GITHUB_TOKEN` environment variable to skip the prompt.

**To skip this notebook altogether**, go straight to Stage 3b: the diffs this notebook
fetched are published on Zenodo and downloaded automatically. Re-fetching them can differ
slightly if a repository has since rewritten its history.

In [4]:
GITHUB_API = "https://api.github.com"
TOKEN = os.environ.get("GITHUB_TOKEN") or getpass.getpass("GitHub token: ").strip()


def api(url, retries=4):
    """One GET. Returns (parsed, status). Handles both rate limits GitHub applies."""
    req = urllib.request.Request(url, headers={
        "Accept": "application/vnd.github+json",
        "User-Agent": "levin-diff-extraction",
        **({"Authorization": f"Bearer {TOKEN}"} if TOKEN else {}),
    })
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(req, timeout=60) as r:
                return json.load(r), 200
        except urllib.error.HTTPError as e:
            h = {k.lower(): v for k, v in e.headers.items()}
            if e.code == 403 and h.get("x-ratelimit-remaining") == "0":
                wait = max(0, int(h.get("x-ratelimit-reset", 0)) - int(time.time())) + 5
                print(f"\n   hourly quota spent; waiting {wait//60}m{wait%60}s")
                time.sleep(wait); continue
            if e.code in (403, 429) and "retry-after" in h:
                # secondary limit: a burst throttle, not exhaustion -- sleep and carry on
                time.sleep(int(h["retry-after"]) + 1); continue
            return None, e.code
        except urllib.error.URLError:
            time.sleep(2 ** attempt)
    return None, 0


quota, _ = api(f"{GITHUB_API}/rate_limit")
if quota:
    core = quota["resources"]["core"]
    print(f"quota {core['remaining']}/{core['limit']}, resets in "
          f"{max(0, core['reset']-int(time.time()))//60}m")
    if core["limit"] <= 60:
        print("\n   Unauthenticated. 1,149 commits will take about 20 hours at 60/hour.")
        print("   Supply a token and re-run this cell.")
else:
    print("could not reach the API")

GitHub token: ··········
quota 5000/5000, resets in 60m


In [5]:
print("verifying each repository is reachable under the mapped name:\n")
resolved = {}
for name in sorted(D[PROJECT].unique()):
    slug = PROJECT_REPO.get(name)
    if not slug:
        continue
    meta, status = api(f"{GITHUB_API}/repos/{slug}")
    if status == 200:
        actual = meta["full_name"]
        moved = "" if actual.lower() == slug.lower() else f"   -> redirects to {actual}"
        resolved[name] = actual
        print(f"   {name:<26}{slug:<36}ok{moved}")
    else:
        print(f"   {name:<26}{slug:<36}HTTP {status}   NOT REACHABLE")

if len(resolved) < D[PROJECT].nunique():
    print("\nSome repositories could not be reached. Their commits will fail; fix the mapping")
    print("above, or accept the loss and let it be reported at the end.")

verifying each repository is reachable under the mapped name:

   ReactiveX-RxJava          ReactiveX/RxJava                    ok
   camel                     apache/camel                        ok
   drools                    apache/incubator-kie-drools         ok   -> redirects to apache/incubator-kie
   elasticsearch             elastic/elasticsearch               ok
   hadoop                    apache/hadoop                       ok
   hbase                     apache/hbase                        ok
   intellij-community        JetBrains/intellij-community        ok
   kotlin                    JetBrains/kotlin                    ok
   orientdb                  orientechnologies/orientdb          ok
   restlet-framework-java    restlet/restlet-framework-java      ok
   spring-framework          spring-projects/spring-framework    ok


## 2. Fetch

One request per commit. The response carries the patch for each changed file, along with the commit
message as recorded in git — which is worth keeping, because the dataset's `comment` column appears
to be preprocessed and the raw text may differ.

**Resumable.** Each commit is appended to a JSONL as it arrives, and a re-run skips what is already
there. This job takes long enough that a dropped connection or a spent quota should not mean
starting again.

**Large commits are truncated by GitHub**, not here: the endpoint returns at most 300 files. The
flag is recorded per commit so Stage 4 can exclude them if truncation matters.

In [6]:
OUT = Path("diffs"); OUT.mkdir(exist_ok=True)
DIFFS = OUT / "levin_diffs.jsonl"

done = set()
if DIFFS.exists():
    with open(DIFFS, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    done.add(json.loads(line)["sha"])
                except json.JSONDecodeError:
                    continue
    print(f"{len(done):,} commits already fetched; resuming")

todo = [(r[ID], r[PROJECT]) for _, r in D.iterrows() if r[ID] not in done]
print(f"{len(todo):,} to fetch")


def fetch_commit(sha, project):
    slug = resolved.get(project) or PROJECT_REPO.get(project)
    if not slug:
        return {"sha": sha, "project": project, "ok": False, "reason": "no repo mapping"}
    data, status = api(f"{GITHUB_API}/repos/{slug}/commits/{sha}")
    if status != 200:
        return {"sha": sha, "project": project, "repo": slug, "ok": False,
                "reason": f"HTTP {status}"}
    files = data.get("files") or []
    return {
        "sha": sha, "project": project, "repo": slug, "ok": True,
        "git_message": (data.get("commit") or {}).get("message", ""),
        "author_date": ((data.get("commit") or {}).get("author") or {}).get("date"),
        "n_files": len(files),
        "truncated": len(files) >= 300,      # the endpoint caps at 300 files
        "stats": data.get("stats") or {},
        "files": [{"filename": f.get("filename"), "status": f.get("status"),
                   "additions": f.get("additions"), "deletions": f.get("deletions"),
                   "patch": f.get("patch")} for f in files],
    }


t0 = time.time()
with open(DIFFS, "a", encoding="utf-8") as out:
    for i, (sha, project) in enumerate(todo, 1):
        rec = fetch_commit(sha, project)
        out.write(json.dumps(rec, ensure_ascii=False) + "\n")
        out.flush()
        if i % 25 == 0 or i == len(todo):
            rate = i / max(1e-9, time.time() - t0)
            left = (len(todo) - i) / max(1e-9, rate)
            print(f"\r   {i:,}/{len(todo):,}   {rate*60:.0f}/min   "
                  f"~{left/60:.0f}m remaining", end="", flush=True)
print(f"\n\ndone in {(time.time()-t0)/60:.1f} minutes")

1,151 to fetch
   1,151/1,151   154/min   ~0m remaining

done in 7.5 minutes


## 3. Coverage

How much of the labelled set survived, and where the losses fall. A loss concentrated in one project
or one class matters more than the same number spread evenly — it would bias what Stage 4 learns.

In [7]:
R = pd.read_json(DIFFS, lines=True)
R = R.drop_duplicates(subset=["sha"], keep="last")
M = D.merge(R, left_on=ID, right_on="sha", how="left", suffixes=("", "_r"))
M["ok"] = M["ok"].fillna(False)

print(f"{int(M.ok.sum()):,} of {len(M):,} commits retrieved "
      f"({M.ok.mean()*100:.1f}%)\n")

print("BY PROJECT\n")
by_p = M.groupby(PROJECT)["ok"].agg(["sum", "count"])
by_p.columns = ["retrieved", "total"]
by_p["%"] = (by_p.retrieved / by_p.total * 100).round(1)
print(by_p.sort_values("%").to_string())

print("\n\nBY LABEL\n")
NAMES = {"c": "corrective", "p": "perfective", "a": "adaptive"}
M["_y"] = M[LABEL].astype(str).str.strip().str.lower().map(NAMES)
by_l = M.groupby("_y")["ok"].agg(["sum", "count"])
by_l.columns = ["retrieved", "total"]
by_l["%"] = (by_l.retrieved / by_l.total * 100).round(1)
print(by_l.to_string())
print("\nIf one class is retrieved at a markedly lower rate, the surviving set is not a random")
print("subset of the labelled data, and Stage 4's comparison against Stage 2 is weakened.")

if (~M.ok).any():
    print("\n\nWHY THE REST FAILED\n")
    print(M.loc[~M.ok, "reason"].fillna("not attempted").value_counts().to_string())

1,151 of 1,151 commits retrieved (100.0%)

BY PROJECT

                        retrieved  total      %
project                                        
ReactiveX-RxJava              100    100  100.0
camel                         103    103  100.0
drools                        114    114  100.0
elasticsearch                 100    100  100.0
hadoop                        112    112  100.0
hbase                         109    109  100.0
intellij-community            109    109  100.0
kotlin                        101    101  100.0
orientdb                      102    102  100.0
restlet-framework-java        101    101  100.0
spring-framework              100    100  100.0


BY LABEL

            retrieved  total      %
_y                                 
adaptive          247    247  100.0
corrective        500    500  100.0
perfective        404    404  100.0

If one class is retrieved at a markedly lower rate, the surviving set is not a random
subset of the labelled data, and Stage 4's

## 4. What the diffs look like

Size distribution matters for Stage 4. CodeBERT reads 512 tokens; a diff touching forty files will
not fit, and how it is shortened is a modelling decision that has to be stated. This shows how often
that decision will bite.

In [8]:
ok = M[M.ok].copy()
ok["n_files"] = ok["n_files"].fillna(0).astype(int)
ok["patch_chars"] = ok["files"].map(
    lambda fs: sum(len(f.get("patch") or "") for f in (fs or [])))
ok["java_files"] = ok["files"].map(
    lambda fs: sum(1 for f in (fs or []) if str(f.get("filename", "")).endswith(".java")))

print("FILES PER COMMIT\n")
print(ok["n_files"].describe(percentiles=[.5, .75, .9, .95, .99]).round(1).to_string())
print(f"\n   commits touching no Java file : {int((ok.java_files == 0).sum())}")
print(f"   truncated by the API at 300   : {int(ok['truncated'].fillna(False).sum())}")

print("\n\nPATCH SIZE IN CHARACTERS\n")
print(ok["patch_chars"].describe(percentiles=[.5, .75, .9, .95, .99]).round(0).to_string())

# ~4 characters per token is the usual rough conversion for code.
approx_tokens = ok["patch_chars"] / 4
for limit in (512, 1024, 2048):
    share = (approx_tokens > limit).mean() * 100
    print(f"\n   roughly {share:.0f}% of diffs exceed {limit} tokens")
print("""
A majority over 512 means truncation is the normal case, not the exception, and the choice of
what to keep -- the first N tokens, changed lines only, per-file truncation -- is a design
decision Stage 4 must state rather than default into.""")

FILES PER COMMIT

count    1151.0
mean        5.5
std        14.1
min         1.0
50%         2.0
75%         5.0
90%        12.0
95%        18.5
99%        43.5
max       300.0

   commits touching no Java file : 1
   truncated by the API at 300   : 1


PATCH SIZE IN CHARACTERS

count      1151.0
mean      11570.0
std       24152.0
min         186.0
50%        3802.0
75%       10516.0
90%       27039.0
95%       44720.0
99%      145006.0
max      231752.0

   roughly 66% of diffs exceed 512 tokens

   roughly 48% of diffs exceed 1024 tokens

   roughly 29% of diffs exceed 2048 tokens

A majority over 512 means truncation is the normal case, not the exception, and the choice of
what to keep -- the first N tokens, changed lines only, per-file truncation -- is a design
decision Stage 4 must state rather than default into.


## 5. Is the dataset's `comment` the raw commit message?

Stage 1 found no `#` characters in the `comment` column across 1,151 messages, which suggested the
export had stripped them. The API returns the message as git stores it, so the two can now be
compared directly.

This matters because Stage 4 trains on one and Stage 5 applies to the other. If they differ
systematically, that is a distribution shift between training and application.

In [9]:
cmp = ok[[MESSAGE, "git_message"]].dropna()
cmp = cmp[cmp["git_message"].astype(str).str.len() > 0]

def norm(s):
    return " ".join(str(s).split()).strip().lower()

exact = (cmp[MESSAGE].map(norm) == cmp["git_message"].map(norm)).mean() * 100
first_line = (cmp[MESSAGE].map(norm) ==
              cmp["git_message"].map(lambda s: norm(str(s).split("\n")[0]))).mean() * 100

print(f"identical after whitespace normalisation      : {exact:.1f}%")
print(f"dataset matches only the git first line       : {first_line:.1f}%")
print()
print(f"'#' in dataset comment : {int(cmp[MESSAGE].astype(str).str.contains('#').sum())}")
print(f"'#' in git message     : {int(cmp['git_message'].astype(str).str.contains('#').sum())}")

print("\n\nTHREE EXAMPLES\n")
for _, r in cmp.head(3).iterrows():
    print(f"   dataset : {str(r[MESSAGE])[:100]}")
    print(f"   git     : {str(r['git_message'])[:100].replace(chr(10), ' / ')}\n")

print("""If the dataset holds only the first line while git holds the full body, Stage 4 has a
choice to make and to state: train on the same truncated text the labellers saw, or on the full
message the model would meet in practice.""")

identical after whitespace normalisation      : 0.0%
dataset matches only the git first line       : 0.0%

'#' in dataset comment : 0
'#' in git message     : 118


THREE EXAMPLES

   dataset : Add marble diagrams for Single operators.--
   git     : Add marble diagrams for Single operators.

   dataset : Change hasException to hasThrowable--
   git     : Change hasException to hasThrowable

   dataset : Trying to extend the Scheduler interface according- to the comments at -19.--
   git     : Trying to extend the Scheduler interface according to the comments at / #19.

If the dataset holds only the first line while git holds the full body, Stage 4 has a
choice to make and to state: train on the same truncated text the labellers saw, or on the full
message the model would meet in practice.


## 6. Save

A single file per commit joining the label, the message as published, the message as git records it,
and the diff. Stage 4 reads this and nothing else.

In [10]:
KEEP = ["sha", "project", "repo", "ok", "reason", "git_message", "author_date",
        "n_files", "truncated", "stats", "files"]
export = M[[ID, PROJECT, MESSAGE, LABEL, "_y"] + [c for c in KEEP if c in M.columns]].copy()
export = export.rename(columns={ID: "commit_id", PROJECT: "project_name",
                                MESSAGE: "dataset_comment", LABEL: "label_code",
                                "_y": "label"})

def clean(v):
    """Null-safe for scalars; lists and dicts pass through. pd.isna on a list returns an array."""
    if isinstance(v, (list, dict)):
        return v
    try:
        return None if pd.isna(v) else v
    except (TypeError, ValueError):
        return v


path = Path("levin_with_diffs.jsonl")
with open(path, "w", encoding="utf-8") as f:
    for _, r in export.iterrows():
        f.write(json.dumps({k: clean(v) for k, v in r.items()},
                           ensure_ascii=False, default=str) + "\n")

summary = {
    "commits_total": int(len(M)),
    "commits_retrieved": int(M.ok.sum()),
    "retrieval_rate": round(float(M.ok.mean()), 4),
    "by_label": {k: int(v) for k, v in M[M.ok].groupby("_y").size().items()},
    "truncated_at_300_files": int(ok["truncated"].fillna(False).sum()),
    "median_files": float(ok["n_files"].median()),
    "median_patch_chars": float(ok["patch_chars"].median()),
}
json.dump(summary, open("stage3_summary.json", "w"), indent=1)
print(json.dumps(summary, indent=1))
print(f"\n   {path} and stage3_summary.json written")
print("""
Stage 4 must use the same split indices saved by Stage 2, restricted to the commits retrieved
here. Comparing a model trained on a subset against Stage 2's numbers on the full set would
flatter whichever had the easier rows.""")

{
 "commits_total": 1151,
 "commits_retrieved": 1151,
 "retrieval_rate": 1.0,
 "by_label": {
  "adaptive": 247,
  "corrective": 500,
  "perfective": 404
 },
 "truncated_at_300_files": 1,
 "median_files": 2.0,
 "median_patch_chars": 3802.0
}

   levin_with_diffs.jsonl and stage3_summary.json written

Stage 4 must use the same split indices saved by Stage 2, restricted to the commits retrieved
here. Comparing a model trained on a subset against Stage 2's numbers on the full set would
flatter whichever had the easier rows.
